In [40]:
!pip install transformers datasets seqeval sentencepiece -q

In [41]:
import pandas as pd
import ast
import torch
import numpy as np
import re
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from torch.utils.data import DataLoader
from collections import defaultdict

print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

CUDA available: True
Device: cuda


In [42]:
df = pd.read_csv("https://docs.google.com/spreadsheets/d/1KXcA0PPOpygla1inEfnTc10FND6DFNL8p8vpQKAX6Tw/export?format=csv")

print("Dataset shape:", df.shape)
print(df.head(3))

Dataset shape: (4000, 6)
  parent_asin  sentence_id                                      sentence_text  \
0  B07DGRVTWF            3  basically a poor implementation of the alexa p...   
1  B07BFPJ6VX            4  support through direct messaging was great no ...   
2  B0B1PYNH8X            1  for echo show 5 i have the 2nd generation echo...   

   rating                                           triplets   category_name  
0     2.0     [["alexa platform", "poor implementation", 0]]  electronics_p2  
1     5.0  [["support through direct messaging", "great",...  electronics_p2  
2     5.0                                                 []  electronics_p2  


In [43]:
SENTIMENT_MAP = {
    0: "negative",
    1: "neutral",
    2: "positive",
    "0": "negative",
    "1": "neutral",
    "2": "positive"
}

def safe_parse(x):
    """Parse triplet string safely"""
    try:
        return ast.literal_eval(str(x))
    except:
        return []

def triplets_to_target(triplets):
    """
    Chuyển list triplets thành chuỗi target cho T5.
    Ví dụ: [["screen", "bad", 0], ["battery", "great", 2]]
    -> "(screen, bad, negative) ; (battery, great, positive)"
    """
    parts = []
    for triplet in triplets:
        if len(triplet) == 3:
            aspect, opinion, sentiment = triplet
            senti_str = SENTIMENT_MAP.get(sentiment, "neutral")
            parts.append(f"({aspect.strip()}, {opinion.strip()}, {senti_str})")
    return " ; ".join(parts)

# Apply parsing
df["triplets"] = df["triplets"].apply(safe_parse)
df = df[df["triplets"].apply(lambda x: len(x) > 0)].reset_index(drop=True)

# Tạo cột target (output của T5)
df["target"] = df["triplets"].apply(triplets_to_target)

# Tạo cột input (input của T5)
df["input"] = "extract triplets: " + df["sentence_text"].astype(str)

print(f"Filtered dataset size: {len(df)}")
print("\nSample input/output:")
for _, row in df.head(3).iterrows():
    print(f"  INPUT : {row['input']}")
    print(f"  TARGET: {row['target']}")
    print()

Filtered dataset size: 2271

Sample input/output:
  INPUT : extract triplets: basically a poor implementation of the alexa platform
  TARGET: (alexa platform, poor implementation, negative)

  INPUT : extract triplets: support through direct messaging was great no language barrier very responsive
  TARGET: (support through direct messaging, great, positive) ; (support through direct messaging, no language barrier, positive) ; (support through direct messaging, very responsive, positive)

  INPUT : extract triplets: i like this ipad cover i like this ipad cover
  TARGET: (ipad cover, like, positive)



In [44]:
train_df, test_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")

# Phân phối sentiment trong train set
from collections import Counter
all_senti = []
for trips in train_df["triplets"]:
    for t in trips:
        if len(t) == 3:
            all_senti.append(SENTIMENT_MAP.get(t[2], "unknown"))
print("Sentiment distribution (train):", Counter(all_senti))

Train: 2043 | Test: 228
Sentiment distribution (train): Counter({'positive': 2248, 'negative': 1063, 'neutral': 162})


In [45]:
MODEL_NAME = "t5-base"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 256

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [46]:
def preprocess(examples):
    """Tokenize input và target cho Seq2Seq training"""
    model_inputs = tokenizer(
        examples["input"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length"
    )


    labels = tokenizer(
        text_target=examples["target"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length"
    )

    # Thay padding token bằng -100 để loss function bỏ qua
    label_ids = labels["input_ids"]
    label_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in label_ids
    ]
    model_inputs["labels"] = label_ids
    return model_inputs


train_dataset = Dataset.from_dict({"input": train_df["input"].tolist(), "target": train_df["target"].tolist()})
test_dataset  = Dataset.from_dict({"input": test_df["input"].tolist(),  "target": test_df["target"].tolist()})

train_dataset = train_dataset.map(preprocess, batched=True, remove_columns=["input", "target"])
test_dataset  = test_dataset.map(preprocess,  batched=True, remove_columns=["input", "target"])

print("Train dataset:", train_dataset)
print("Test dataset: ", test_dataset)

Map:   0%|          | 0/2043 [00:00<?, ? examples/s]

Map:   0%|          | 0/228 [00:00<?, ? examples/s]

Train dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2043
})
Test dataset:  Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 228
})


In [47]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model = model.to(device)

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {params:,}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Trainable parameters: 222,903,552


In [48]:
def parse_generated_triplets(text):
    """
    Parse chuỗi sinh ra từ T5 thành list of (aspect, opinion, sentiment).
    Input:  "(screen, bad, negative) ; (battery, great, positive)"
    Output: [("screen", "bad", "negative"), ("battery", "great", "positive")]
    """
    triplets = []

    pattern = r'\(([^,]+),\s*([^,]+),\s*(positive|negative|neutral)\)'
    matches = re.findall(pattern, text.lower())
    for m in matches:
        aspect   = m[0].strip()
        opinion  = m[1].strip()
        sentiment = m[2].strip()
        triplets.append((aspect, opinion, sentiment))
    return triplets


def compute_triplet_f1(pred_triplets_list, gold_triplets_list):
    """
    Tính Precision, Recall, F1 trên tập toàn bộ test set.
    Mỗi triplet là một tuple (aspect, opinion, sentiment).
    """
    tp = fp = fn = 0
    for preds, golds in zip(pred_triplets_list, gold_triplets_list):
        pred_set = set(preds)
        gold_set = set(golds)
        tp += len(pred_set & gold_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    return {"precision": round(precision, 4),
            "recall":    round(recall,    4),
            "f1":        round(f1,        4)}


def compute_metrics(eval_pred):
    """
    Hàm compute_metrics đã sửa lỗi OverflowError
    """
    predictions, labels = eval_pred


    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Giải mã labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Parse thành triplets
    pred_triplets = [parse_generated_triplets(p) for p in decoded_preds]
    gold_triplets = [parse_generated_triplets(g) for g in decoded_labels]

    scores = compute_triplet_f1(pred_triplets, gold_triplets)
    return scores


print("Test parse_generated_triplets:")
sample = "(screen, bad, negative) ; (battery, great, positive)"
print(" ", parse_generated_triplets(sample))

Test parse_generated_triplets:
  [('screen', 'bad', 'negative'), ('battery', 'great', 'positive')]


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

total_train_steps = (len(train_dataset) // 16) * 5
warmup_steps = int(total_train_steps * 0.1)

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_absa_output",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-4,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.540551,0.391926,0.372000,0.377000,0.374500
2,0.315031,0.371514,0.380200,0.420800,0.399500
3,0.200209,0.381085,0.443500,0.439900,0.441700
4,0.132949,0.416725,0.435100,0.439900,0.437500
5,0.098594,0.425958,0.434600,0.453600,0.443900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=640, training_loss=0.37511880323290825, metrics={'train_runtime': 645.4315, 'train_samples_per_second': 15.827, 'train_steps_per_second': 0.992, 'total_flos': 1555126213017600.0, 'train_loss': 0.37511880323290825, 'epoch': 5.0})

In [50]:
# Đánh giá cuối cùng trên test set
results = trainer.evaluate(test_dataset)

print(f"  Precision : {results.get('eval_precision', 0):.4f}")
print(f"  Recall    : {results.get('eval_recall', 0):.4f}")
print(f"  F1 Score  : {results.get('eval_f1', 0):.4f}")
print(f"  Loss      : {results.get('eval_loss', 0):.4f}")


  Precision : 0.4346
  Recall    : 0.4536
  F1 Score  : 0.4439
  Loss      : 0.4260


In [51]:
# sinh ra predictions và tính metrics theo từng nhóm sentiment
model.eval()
model.to(device)

all_preds = []
all_golds = []

for _, row in test_df.iterrows():
    input_text = row["input"]
    gold_text  = row["target"]

    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=MAX_INPUT_LEN,
        truncation=True
    ).input_ids.to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_length=MAX_TARGET_LEN,
            num_beams=4,               # Beam search để chất lượng tốt hơn
            early_stopping=True
        )

    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    all_preds.append(parse_generated_triplets(pred_text))
    all_golds.append(parse_generated_triplets(gold_text))

# Overall F1
overall = compute_triplet_f1(all_preds, all_golds)
#DETAILED TEST SET EVALUATION
print(f"  Precision : {overall['precision']:.4f}")
print(f"  Recall    : {overall['recall']:.4f}")
print(f"  F1 Score  : {overall['f1']:.4f}")


for senti in ["positive", "negative", "neutral"]:
    filtered_pred = [[t for t in preds if t[2] == senti] for preds in all_preds]
    filtered_gold = [[t for t in golds if t[2] == senti] for golds in all_golds]
    score = compute_triplet_f1(filtered_pred, filtered_gold)
    print(f"  [{senti:8s}] P={score['precision']:.4f}  R={score['recall']:.4f}  F1={score['f1']:.4f}")

  Precision : 0.4607
  Recall    : 0.4481
  F1 Score  : 0.4543
  [positive] P=0.5179  R=0.5022  F1=0.5099
  [negative] P=0.3810  R=0.3871  F1=0.3840
  [neutral ] P=0.0000  R=0.0000  F1=0.0000


In [52]:
def extract_triplets(sentence, num_beams=4):
    """
    Sinh ra triplets (aspect, opinion, sentiment) từ một câu.

    Args:
        sentence: câu tiếng Anh cần phân tích
        num_beams: số beam trong beam search

    Returns:
        List of (aspect, opinion, sentiment)
    """
    model.eval()
    input_text = "extract triplets: " + sentence
    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=MAX_INPUT_LEN,
        truncation=True
    ).input_ids.to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_length=MAX_TARGET_LEN,
            num_beams=num_beams,
            early_stopping=True
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    triplets = parse_generated_triplets(generated)
    return generated, triplets


test_sentences = [
    "battery is great but screen is bad",
    "support through direct messaging was great no language barrier very responsive",
    "the sound quality is excellent but the build feels cheap",
    "basically a poor implementation of the alexa platform",
    "I love this keyboard, it is very comfortable and silent"
]


for sent in test_sentences:
    generated_str, triplets = extract_triplets(sent)
    print(f"\nInput   : {sent}")
    print(f"Generated: {generated_str}")
    print(f"Triplets: {triplets}")


Input   : battery is great but screen is bad
Generated: (battery, great, positive) ; (screen, bad, negative)
Triplets: [('battery', 'great', 'positive'), ('screen', 'bad', 'negative')]

Input   : support through direct messaging was great no language barrier very responsive
Generated: (support through direct messaging, great, positive) ; (support through direct messaging, very responsive, positive)
Triplets: [('support through direct messaging', 'great', 'positive'), ('support through direct messaging', 'very responsive', 'positive')]

Input   : the sound quality is excellent but the build feels cheap
Generated: (sound quality, excellent, positive) ; (build, feels cheap, negative)
Triplets: [('sound quality', 'excellent', 'positive'), ('build', 'feels cheap', 'negative')]

Input   : basically a poor implementation of the alexa platform
Generated: (alexa platform, poor implementation, negative)
Triplets: [('alexa platform', 'poor implementation', 'negative')]

Input   : I love this key

In [55]:
#SAMPLE PREDICTIONS vs GROUND TRUTH
n_show = 10
correct = 0

for i, (_, row) in enumerate(test_df.sample(n_show).iterrows()):
    gen_str, pred_trips = extract_triplets(row["sentence_text"])
    gold_trips = parse_generated_triplets(row["target"])

    pred_set = set(pred_trips)
    gold_set = set(gold_trips)
    match = pred_set == gold_set
    if match:
        correct += 1

    status = "V" if match else "X"
    print(f"\n[{status}] {row['sentence_text']}")
    print(f"  GOLD : {gold_trips}")
    print(f"  PRED : {pred_trips}")

print(f"\nExact match: {correct}/{n_show} = {correct/n_show:.1%}")


[V] otherwise sound quality is good
  GOLD : [('sound quality', 'good', 'positive')]
  PRED : [('sound quality', 'good', 'positive')]

[X] i dont regret reading this book even though the authors writing style took a [GENERIC_NOUN] of adjusting and the ending left me wanting seriously wanting
  GOLD : [('book', 'dont regret', 'positive'), ('writing style', 'took a bit of adjusting', 'neutral'), ('ending', 'left me wanting', 'negative')]
  PRED : [('authors writing style', 'took a adjusting', 'negative'), ('ending', 'left me wanting', 'negative')]

[X] great purchase i get a [GENERIC_NOUN] of advertising mail and this little guy works wonderfully now i use it and dont need to shred all of this garbage
  GOLD : [('purchase', 'great', 'positive'), ('little guy', 'works wonderfully', 'positive'), ('advertising mail', 'dont need to shred', 'positive')]
  PRED : [('little guy', 'works wonderfully', 'positive')]

[V] i thought the long run and the armageddon blues were spectacular when they c